# Who has the ball? Possession from a match video

Upload a football match video and get, five times a second, which team has the ball: the possession split, a timeline of every spell, the video with the teams and the ball marked, and the raw numbers.

**Before you start:** Runtime > Change runtime type > T4 GPU. It runs without a GPU too, only much more slowly.

**بالعربي:** ارفع فيديو المباراة وشغّل الخلايا بالترتيب. هتطلع لك نسب الاستحواذ، وخط زمني لكل فترة استحواذ، والفيديو وعليه علامات الفريقين والكرة. فعّل الـ GPU الأول من Runtime ثم Change runtime type.

Code and method: https://github.com/amr7772/possession-from-tracking

## 1. Get the code (about a minute)

In [ ]:
!nvidia-smi -L || echo "No GPU found: Runtime > Change runtime type > T4 GPU"
!test -d /content/possession-from-tracking || git clone -q https://github.com/amr7772/possession-from-tracking.git /content/possession-from-tracking
%cd /content/possession-from-tracking
!pip -q install -r requirements-video.txt

## 2. Upload the video

A short clip uploads in seconds. A full match is a few gigabytes: put it in your Google Drive first, set USE_DRIVE to True and give its path.

In [ ]:
from pathlib import Path
from google.colab import files

USE_DRIVE = False                          # True for a long video already in Google Drive
DRIVE_PATH = "/content/drive/MyDrive/match.mp4"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = Path(DRIVE_PATH)
else:
    uploaded = files.upload()
    VIDEO = Path(next(iter(uploaded)))
print("Video:", VIDEO)

## 3. Analyse

TEAMS gives each team with its shirt colour: "Zamalek=white,Al Ahly=red". A colour word (white, red, navy, lime...) or a hex code is enough; it only has to be close. Leave TEAMS empty and the teams are named after their shirts.

To try it quickly, set DURATION = 120 (two minutes). To rename the teams afterwards, change TEAMS and run this cell again: the detections are saved, so only the quick steps run twice.

In [ ]:
from possession.video import analyse

TEAMS = ""              # e.g. "Zamalek=white,Al Ahly=red"; empty: named after the shirts
START = 0               # second to start from
DURATION = None         # seconds to analyse; None for the whole video
MODEL = "yolo26m.pt"    # downloaded on first use

summary = analyse(VIDEO, Path("out"), weights=MODEL, start=START, duration=DURATION, teams=TEAMS or None)

## 4. Results

In [ ]:
from IPython.display import HTML, Video, display

display(HTML(Path("out/report.html").read_text()))
clip = Path("out/annotated.mp4")
if clip.exists() and clip.stat().st_size < 50e6:
    display(Video(str(clip), embed=True, width=800))
else:
    print("The annotated video is too big to show here; download it with the next cell.")

In [ ]:
import shutil
shutil.make_archive("/content/possession_results", "zip", "out")
files.download("/content/possession_results.zip")

**What it does not do yet.** It has not been scored against an event log, so read its numbers as a first estimate. Distances are measured in the picture, not on the pitch, so a ball passing in front of a player can count as near him. Goalkeepers and referees are never given the ball. Replays count as live play unless the grass leaves the picture.